
# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table


In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
%run ../00-common/05.data-quality-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


#### Step 1 - Read bronze `circuits` table

In [0]:
circuits_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))


#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)


#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)


In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)


#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)


#### Step 6 - Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])


#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)


#### step 8 - Data Quality Checks

In [0]:
dq_checks = [
    # not_null
    {"name": "circuit_id_not_null",     "type": "not_null", "column": "circuit_id",    "severity": "critical"},
    {"name": "circuit_name_not_null",   "type": "not_null", "column": "circuit_name",  "severity": "warning"},
    {"name": "country_not_null",        "type": "not_null", "column": "country",       "severity": "warning"},
    {"name": "latitude_not_null",       "type": "not_null", "column": "latitude",      "severity": "warning"},
    {"name": "longitude_not_null",      "type": "not_null", "column": "longitude",     "severity": "warning"},
    # unique
    {"name": "circuit_id_unique",       "type": "unique",   "columns": ["circuit_id"], "severity": "critical"},
    # min_rows
    {"name": "min_one_row",             "type": "min_rows", "threshold": 1,            "severity": "warning"},
    # value_range
    {"name": "latitude_range",          "type": "value_range", "column": "latitude",  "min": -90,  "max": 90,  "severity": "warning"},
    {"name": "longitude_range",         "type": "value_range", "column": "longitude", "min": -180, "max": 180, "severity": "warning"},
]

run_dq_checks(
    df              = circuits_final_df,
    checks          = dq_checks,
    table_name      = silver_table,
    batch_id        = v_batch_id,
    raise_on_critical = True
)

#### step 9 - DELTA CHECK CONSTRAINTS

In [0]:
apply_delta_constraints(
    table_name  = silver_table,
    constraints = [
        {"name": "chk_circuit_id_not_null", "condition": "circuit_id IS NOT NULL"},
        {"name": "chk_latitude_range",      "condition": "latitude BETWEEN -90 AND 90"},
        {"name": "chk_longitude_range",     "condition": "longitude BETWEEN -180 AND 180"},
    ]
)


#### Step 10 - Write the transformed data to silver `circuits` table

In [0]:
write_to_silver(
    input_df = circuits_final_df,
    target_table = silver_table,
    merge_condition = "t.circuit_id = s.circuit_id",
    columns_to_update = [
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"  
    ]
)